In [ ]:

from __future__ import annotations

from io import StringIO
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://www.football-data.co.uk/mmz4281/{season}/{division}.csv"

# (division code, human-readable league name)
LEAGUES = [
    ("SP1", "La Liga"),
    ("E0", "Premier League"),
    ("D1", "Bundesliga"),
    ("I1", "Serie A"),
    ("F1", "Ligue 1"),
]

# football-data season codes: 2122 = 2021/22 ... 2526 = 2025/26
SEASONS = ["2122", "2223", "2324", "2425", "2526"]

REQUIRED_COLUMNS = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]

# Preferred bookmakers, in order. For each match, the first source with all
# three odds present becomes the unified odds_home_win/draw/away_win columns.
ODDS_SOURCES = [
    ("b365", ("B365H", "B365D", "B365A")),
    ("pinnacle_close", ("PSCH", "PSCD", "PSCA")),
    ("market_avg", ("AvgH", "AvgD", "AvgA")),
]

# Kept alongside the unified columns for reference.
EXTRA_ODDS_COLUMNS = {
    "AvgH": "market_avg_home_win",
    "AvgD": "market_avg_draw",
    "AvgA": "market_avg_away_win",
}


In [ ]:
def season_label(code: str) -> str:
    """Convert a football-data season code to a readable label: '2425' -> '2024-25'."""
    start = int(code[:2])
    start += 2000 if start < 90 else 1900
    return f"{start}-{str(start + 1)[-2:]}"


def download_division(season: str, division: str) -> pd.DataFrame:
    url = BASE_URL.format(season=season, division=division)
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    content = response.content
    for encoding in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            text = content.decode(encoding)
            break
        except UnicodeDecodeError:
            continue
    return pd.read_csv(StringIO(text), on_bad_lines="skip")

    

In [ ]:
data = download_division("2526", "SP1")
data.head(5)

In [ ]:
%pip install statsbombpy

In [ ]:
from statsbombpy import sb

In [ ]:
comps = sb.competitions()
comps[comps["competition_name"] == "La Liga"]